In [372]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import keras
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from keras.layers import Flatten, Dense, Conv2D, MaxPooling2D
from keras.models import Sequential

In [215]:
daataset = keras.datasets.fashion_mnist.load_data()

In [297]:
(x_train, y_train), (x_test, y_test) = dataset

In [298]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train, 10).astype('float32')
y_test = to_categorical(y_test,10).astype('float32')

In [299]:
print(x_train.shape, x_train.dtype)
print(x_test.shape, x_test.dtype)
print(y_train.shape, y_train.dtype)
print(y_test.shape, y_test.dtype)

(60000, 28, 28) float32
(10000, 28, 28) float32
(60000, 10) float32
(10000, 10) float32


In [403]:
class Keras_model:
    def __init__(self):
        self.model = None
        self.stopping = None
        self.result__train_accuracy = None
        self.result_train_loss = None
        self.result_test_accuracy = None
        self.result_test_loss = None
        
    def buildingmodel(self):
        model = keras.Sequential([
            keras.layers.Input(shape = (28, 28, 1)),
            keras.layers.Conv2D(filters = 32, kernel_size = (3, 3), padding = 'same', activation = 'relu', strides = 2),
            keras.layers.MaxPooling2D(pool_size = (2, 2)),
            keras.layers.Conv2D(filters = 64, kernel_size = (2, 2), activation = 'relu'),
            keras.layers.MaxPooling2D(pool_size = (2, 2)),
            keras.layers.Conv2D(filters = 64, kernel_size = (2, 2), activation = 'relu'),
            keras.layers.Flatten(),
            keras.layers.Dense(units = 64, activation = 'relu', kernel_regularizer = regularizers.L2(0.001)),
            keras.layers.Dense(units = 10, activation = 'softmax')
        ])
        model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])
        self.model = model

    def model_stopping(self):
        stopping_ = keras.callbacks.EarlyStopping(
            monitor="accuracy",
            min_delta=0,
            patience=4,
            verbose=1,
            mode="max",
            baseline=None,
            restore_best_weights=True,
            start_from_epoch=0
            )
        self.stopping = stopping_

    def model_fit(self,xtrain, ytrain, batch, nepoch):
        self.model.fit(xtrain, ytrain, batch_size = batch, epochs = nepoch, verbose = 0, callbacks = [self.stopping])
        return print('Done!')

    def train_evaluate(self, xtrain, ytrain):
        train_loss, train_accuracy = self.model.evaluate(xtrain, ytrain, verbose = 0)
        self.result_train_accuracy = train_accuracy
        self.result_train_loss = train_loss
        return print(f'Тренировочная ошибка: {train_loss}, Тренировочная точность: {train_accuracy}')

    def test_evaluate(self, xtest, ytest):
        test_loss, test_accuracy = self.model.evaluate(xtest, ytest, verbose = 0)
        self.result_test_accuracy = test_accuracy
        self.result_test_loss = test_loss
        return print(f'Тестовая ошибка: {test_loss}, Тестовая точность: {test_accuracy}')

    def model_result(self):
        print(f'Результат точности тренировочной и  тестовой прогонки : {self.result_train_accuracy:.4f} / {self.result_test_accuracy:.4f} ')
        print(f'Результат ошибки тренировочной и тестовой прогонки: {self.result_train_loss:.4f} / {self.result_test_loss:.4f}')

In [404]:
Model_Keras = Keras_model()
Model_Keras.buildingmodel()
Model_Keras.model_stopping()

In [405]:
#                    X_train, ytrain, n_batches, n epoch  
Model_Keras.model_fit(x_train, y_train, 32, 10)

Restoring model weights from the end of the best epoch: 10.
Done!


In [406]:
Model_Keras.train_evaluate(x_train, y_train)

Тренировочная ошибка: 0.21574941277503967, Тренировочная точность: 0.927649974822998


In [407]:
Model_Keras.test_evaluate(x_test, y_test)

Тестовая ошибка: 0.2923305034637451, Тестовая точность: 0.9043999910354614


In [432]:
print(x_train.shape, x_train.dtype)
print(x_test.shape, x_test.dtype)
print(y_train.shape, y_train.dtype)
print(y_test.shape, y_test.dtype)

(60000, 28, 28) float32
(10000, 28, 28) float32
(60000, 10) float32
(10000, 10) float32


In [607]:
def ola(xtrain, ytrain, xtest, ytest, batch):
    xtrain = xtrain.astype('float32') / 255
    xtest = xtest.astype('float32') / 255
    xtrain = torch.tensor(xtrain).unsqueeze(1)
    xtest = torch.tensor(xtest).unsqueeze(1)

    if ytrain.ndim == 2 and ytrain.shape[1] > 1:
        ytrain = ytrain.argmax(axis=1)
        ytest = ytest.argmax(axis=1)
    ytrain = torch.tensor(ytrain, dtype = torch.long)
    ytest = torch.tensor(ytest, dtype = torch.long)


    train = DataLoader(TensorDataset(xtrain, ytrain), batch_size = batch, shuffle = True)
    test = DataLoader(TensorDataset(xtest, ytest), batch_size = batch, shuffle = False)
    print(xtrain.shape, xtrain.dtype )
    print(xtest.shape, xtest.dtype )
    print(ytrain.shape, ytrain.dtype)
    print(ytest.shape, ytest.dtype)
    return train, test

traindataloader, testdataloader = ola(x_train, y_train, x_test, y_test, batchik)

torch.Size([60000, 1, 28, 28]) torch.float32
torch.Size([10000, 1, 28, 28]) torch.float32
torch.Size([60000]) torch.int64
torch.Size([10000]) torch.int64


In [615]:
class Torch_Model(nn.Module):
    def __init__(self):
        super(Torch_Model, self).__init__()
        self.svert_sloi1 = nn.Conv2d(in_channels = 1, out_channels = 16, kernel_size = 2)
        self.activator1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        self.svert_sloi2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 2, padding = 1)
        self.activator2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2 ,2)
        self.svert_sloi3 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 2, padding = 1)
        self.activator3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        self.poln = nn.Flatten()
        self.poln_sloi1 = nn.Linear(1024, 512)
        self.poln_sloi2 = nn.Linear(512, 10)


    def forward(self, x):
        out = self.svert_sloi1(x)
        out = self.activator1(out)
        out = self.pool1(out)
        #print(out.shape)
        out = self.svert_sloi2(out)
        out = self.activator2(out)
        out = self.pool2(out)
        #print(out.shape)
        out = self.svert_sloi3(out)
        out = self.activator3(out)
        out = self.pool3(out)
        #print(out.shape)
        out = self.poln(out)
        #print(out.shape)
        out = self.poln_sloi1(out)
        out = self.poln_sloi2(out)

        return out

Model_Torch = Torch_Model()
lert = 0.0001
los_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(Model_Torch.parameters(), lr = lert)
batchik = 64
epoha = 5

In [608]:
def training_network(model, loader, func_loss, optimizer, ep):
    model.train()
    n = len(loader)

    for epoch in range(ep):
        train_loss = 0
        for batch, (images, labels) in enumerate(loader):
            optimizer.zero_grad()
            predict = model(images)
            loss = func_loss(predict, labels)

            loss.backward()
            optimizer.step()


            train_loss += loss.item()
            
        print(f'Эпоха: {epoch + 1} / {ep }, Ошибка {train_loss / n}')

In [620]:
training_network(Model_Torch, traindataloader, loss_fn, optimizer, epoha)

Эпоха: 1 / 5, Ошибка 0.833502145273599
Эпоха: 2 / 5, Ошибка 0.7735914531420035
Эпоха: 3 / 5, Ошибка 0.7424171938697921
Эпоха: 4 / 5, Ошибка 0.720058007153875
Эпоха: 5 / 5, Ошибка 0.7027935689128538


In [610]:
def testing_network(loader, model, loss_fn):
    model.eval()
    num_batch = len(loader)
    size = len(loader.dataset)
    val_loss = 0.0
    correct = 0.0
    total = 0.0

    with torch.no_grad():
        for images, labels in loader:
            predictt = model(images)
            losss = loss_fn(predictt, labels)
            val_loss = losss.item()
            predict_label = predictt.argmax(dim = 1)
            correct += (predictt.argmax(dim = 1) == labels).sum().item()
            total += labels.size(0)
    print(f'loss {val_loss / num_batch:.4f}, accuracy {correct / total:.4f} ')

In [621]:
testing_network(testdataloader, Model_Torch, loss_fn)

loss 0.0021, accuracy 0.7239 


In [605]:
Model_Keras.model_result()

Результат точности тренировочной и  тестовой прогонки : 0.9276 / 0.9044 
Результат ошибки тренировочной и тестовой прогонки: 0.2157 / 0.2923
